# Football Market Value Prediction
**TDDE64 Sports Analytics — Linköping University**

## Project overview

The goal is to predict Premier League players' market values using only on-field performance statistics, then benchmark those predictions against Transfermarkt's crowd-sourced values and actual transfer fees.

This follows the methodology of *"Data-Driven Models for Predicting Field Player Market Value in European Football"* (IEEE doc 11264761), which reports R² > 0.80 using Random Forest and XGBoost across European leagues.

### Pipeline
```
API-Football (stats) ──┐
                       ├── join.py ── features.py ── models.py
Transfermarkt (values) ┘
```

### Data sources
- **API-Football**: per-match aggregated stats (goals, assists, passes, tackles, etc.) for all PL players, queried by team to work around the free-tier page limit
- **Transfermarkt**: historical market valuations (target variable), club metadata, and actual transfer fees (for benchmarking)

### Important modeling choices
- **Exclude goalkeepers** — their value drivers are completely different from field players
- **Log-transform the target**: `y = log1p(market_value_eur)` — market values are heavily right-skewed
- **Temporal train/test split**: train on past seasons, test on the most recent (no data leakage)
- **Position-aware models**: separate DEF / MID / FWD models, compared against a single global model

In [ ]:
import logging
import sys
sys.path.insert(0, '..')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker

logging.basicConfig(level=logging.INFO, format='%(levelname)s %(message)s')

%matplotlib inline
plt.rcParams['figure.figsize'] = (10, 5)
plt.rcParams['figure.dpi'] = 120

---
## 1. Load data

We have two independent data sources that need to be loaded and filtered to Premier League players before they can be joined.

### 1a. Transfermarkt
`load_transfermarkt()` reads four CSVs from `data/raw/transfermarkt/` and filters each to PL-relevant rows:
- **clubs**: all clubs with `domestic_competition_id == 'GB1'` (Premier League competition code)
- **valuations**: all historical market value snapshots recorded while a player was at a PL club — this is our **target variable**
- **players**: all players who appear in the PL valuations (i.e. anyone who has ever played in the PL, not just current squad members)
- **transfers**: all transfers where at least one club is a PL club — used later for benchmarking predictions against real fees

### 1b. API-Football
`fetch_api_football()` queries the cached JSON files in `data/raw/api_football/`. On first run it hits the network; on subsequent runs it reads from disk, so the 100 req/day limit is only consumed once.

We query by **team** (not by league) to avoid the free-tier page-3 cap on the league endpoint. Each team's squad fits in 1–3 pages, and we have 22 PL team IDs across the 3 seasons.

In [ ]:
from src.ingest import load_transfermarkt, fetch_api_football

tm = load_transfermarkt()
api_players = fetch_api_football(seasons=[2022])  # add 2023, 2024 once fetched

print(f"Transfermarkt — clubs: {len(tm.clubs)}  players: {len(tm.players)}  "
      f"valuations: {len(tm.valuations)}  transfers: {len(tm.transfers)}")
print(f"API-Football  — player-records: {len(api_players)}")

---
## 2. Join

API-Football and Transfermarkt use completely different internal player IDs, so we match them manually using player identity attributes.

### Matching strategy (three-step cascade)

1. **Exact match on full name + date of birth** — the most reliable signal. Both names are normalized first: accents stripped via `unidecode`, lowercased, punctuation removed. This catches most mainstream players.

2. **Exact match on last name + date of birth** — handles cases where API-Football stores a compound last name (e.g. *"Danjuma Adam Groeneveld"*) that Transfermarkt splits differently. DOB acts as a hard disambiguator so last-name collisions are rare.

3. **Fuzzy match on full name, same DOB** — uses `rapidfuzz.token_sort_ratio` (threshold ≥ 80) to catch spelling differences and name ordering variations. DOB must still match exactly, limiting false positives.

Players with no match are dropped — they have no market value to predict against.

### Expected match rate
Target is 80–90% per the project plan. Many unmatched players are fringe squad / academy players who appear in the API but have never had a Transfermarkt valuation recorded.

In [ ]:
from src.join import build_crosswalk, build_joined_dataset

crosswalk = build_crosswalk(api_players, tm.players)

print("Match method breakdown:")
print(crosswalk['match_method'].value_counts().to_string())
print(f"\nOverall match rate: {crosswalk['tm_player_id'].notna().mean():.1%}")

### Attaching market values

For each matched player-season, we snap the Transfermarkt valuation to the **closest date to July 1 of season end** (e.g. July 1 2023 for the 2022/23 season). This gives us the market value at the point when the season's stats are complete.

Players who transferred mid-season appear multiple times in the API data (one row per club). We **sum their stats** across all clubs before computing features — a player who scored 5 goals at club A and 3 at club B contributed 8 goals to that season.

In [ ]:
joined = build_joined_dataset(api_players, tm)
print(f"Joined dataset: {len(joined)} rows (player-season-club entries)")
print(f"Unique player-seasons: {joined.drop_duplicates(['api_player_id','season']).shape[0]}")
print(f"Unique players: {joined['api_player_id'].nunique()}")
joined.head(3)

---
## 3. Exploratory Data Analysis

Before building models, we need to understand the distribution of the target variable and the relationships between features. This section answers three key questions:

1. Is the market value distribution skewed, and does log-transformation fix it?
2. Do different positions have systematically different value distributions?
3. Which performance features correlate most strongly with market value?

In [ ]:
from src.features import build_stage1, _aggregate_multi_club, _add_age, _normalize_position

s1 = build_stage1(joined)
print(f"Stage 1 dataset: {len(s1)} player-seasons, {s1.shape[1]} columns")

### 3a. Market value distribution

Football market values are well-known to follow a heavy right tail — a handful of elite players (Haaland, Salah, etc.) are worth 10–20× the median. This violates the normality assumption of linear regression and makes RMSE misleading in raw EUR.

The fix is `log1p(market_value_eur)`, which compresses the tail and makes the distribution roughly normal. All models train on this log-transformed target. We report metrics in both log-space (R², RMSE_log) and back-transformed EUR (MAE_eur) for interpretability.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].hist(s1['market_value_in_eur'] / 1e6, bins=40, color='steelblue', edgecolor='white')
axes[0].set_xlabel('Market value (€M)')
axes[0].set_title('Raw market value — heavy right tail')

axes[1].hist(s1['log_market_value'], bins=40, color='steelblue', edgecolor='white')
axes[1].set_xlabel('log1p(market value)')
axes[1].set_title('Log-transformed target — approximately normal')

plt.tight_layout()
plt.show()

print(f"Raw:  mean=€{s1['market_value_in_eur'].mean()/1e6:.1f}M  median=€{s1['market_value_in_eur'].median()/1e6:.1f}M  max=€{s1['market_value_in_eur'].max()/1e6:.0f}M")

### 3b. Market value by position

Forwards tend to command higher market values than defenders and midfielders — goals are the most visible contribution and easier for scouts to quantify. This position-specific value difference is one reason the reference paper trains **separate models per position** rather than a single global model.

If we pooled all positions, the model would need to simultaneously learn that "10 goals" means something very different for a striker vs. a centre-back. Separate models let each position's features speak to its own value drivers.

In [ ]:
s1_pos = s1.copy()
s1_pos['position'] = s1[[c for c in s1.columns if c.startswith('pos_')]].idxmax(axis=1).str.replace('pos_', '')

fig, ax = plt.subplots(figsize=(9, 4))
for pos, grp in s1_pos.groupby('position'):
    ax.hist(grp['market_value_in_eur'] / 1e6, bins=30, alpha=0.55, label=pos)
ax.set_xlabel('Market value (€M)')
ax.set_title('Market value distribution by position')
ax.legend()
plt.tight_layout()
plt.show()

s1_pos.groupby('position')['market_value_in_eur'].agg(['median','mean','max']).map(lambda x: f'€{x/1e6:.1f}M')

### 3c. Age curve

Market value follows an **inverted U-shape with age** — players peak in their mid-to-late 20s and decline steeply after 30. This reflects both current performance and future sell-on value.

In Stage 3 features we add `age²` to let linear models capture this curvature. Tree models (RF, XGBoost) can learn it automatically through splits.

In [ ]:
fig, ax = plt.subplots(figsize=(9, 4))
for pos, grp in s1_pos.groupby('position'):
    ax.scatter(grp['age'], grp['market_value_in_eur'] / 1e6, alpha=0.35, s=18, label=pos)
ax.set_xlabel('Age at season end')
ax.set_ylabel('Market value (€M)')
ax.set_title('Age vs market value — inverted U-shape expected')
ax.legend()
plt.tight_layout()
plt.show()

### 3d. Feature correlations

A Pearson correlation heatmap gives a quick read of which Stage 1 features are most linearly associated with `log_market_value`. Minutes and appearances are expected to correlate strongly with value — players worth more get more game time. Goals and assists should also be positive.

Age is expected to have a weak or near-zero linear correlation because the relationship is non-linear (inverted U).

In [ ]:
feat_cols = ['age', 'minutes', 'appearances', 'goals', 'assists',
             'yellow_cards', 'red_cards', 'log_market_value']
corr = s1[feat_cols].corr()

fig, ax = plt.subplots(figsize=(8, 6))
im = ax.imshow(corr, vmin=-1, vmax=1, cmap='RdBu_r')
plt.colorbar(im, ax=ax)
ax.set_xticks(range(len(feat_cols)))
ax.set_yticks(range(len(feat_cols)))
ax.set_xticklabels(feat_cols, rotation=45, ha='right')
ax.set_yticklabels(feat_cols)
for i in range(len(feat_cols)):
    for j in range(len(feat_cols)):
        ax.text(j, i, f'{corr.iloc[i, j]:.2f}', ha='center', va='center', fontsize=7)
ax.set_title('Pearson correlation — Stage 1 features vs log market value')
plt.tight_layout()
plt.show()

### 3e. Top 10 most valuable players

A sanity check — the highest-valued players in our dataset should be recognisable names. If we see obvious errors here (e.g. unknown reserve players at the top), it suggests a join error or valuation date mismatch.

In [ ]:
agg = _normalize_position(_add_age(_aggregate_multi_club(joined)))
top10 = (agg.nlargest(10, 'market_value_in_eur')
           [['firstname', 'lastname', 'position_group', 'age', 'goals', 'assists', 'minutes', 'market_value_in_eur']]
           .assign(market_value_in_eur=lambda d: d['market_value_in_eur'].apply(lambda x: f'€{x/1e6:.0f}M'))
           .reset_index(drop=True))
top10

---
## 4. Feature engineering

We mirror the three-stage experimental design from the reference paper. Running all models at each stage lets us isolate the marginal contribution of each feature group.

### Stage 1 — Basic (~10 features)
Age, one-hot position (DEF / MID / FWD), minutes played, appearances, goals, assists, yellow cards, red cards.
These are the simplest objective stats — anything a casual fan would know about a player.

### Stage 2 — Expanded (~22 features)
Stage 1 plus **per-90-minute versions** of all counting stats (goals/90, assists/90, passes/90, etc.) and rate stats (pass accuracy, dribble success %, duels won %, shot accuracy).
Per-90 stats normalise for playing time — a player who scores 10 goals in 900 minutes is equivalent to one who scores 5 in 450. This is the standard in modern football analytics.

> **Note on linear models in Stage 2**: per-90 stats can be extreme for players with very few minutes (1 goal in 45 mins = 2.0 goals/90). With 100+ correlated features for small position subsets, unregularised linear regression completely breaks down. This is visible in the Stage 2 results and is an interesting finding to discuss in the report.

### Stage 3 — Domain-informed (~30+ features)
Stage 2 plus `age²` (captures the non-linear age curve), position-relative percentile ranks (is this player's goal rate in the top 25% for their position?), and — once multiple seasons are loaded — **Δ-stats** (year-over-year change in goals, assists, minutes) which signal trajectory.

In [ ]:
from src.features import build_stage1, build_stage2, build_stage3

s1 = build_stage1(joined)
s2 = build_stage2(joined)
s3 = build_stage3(joined)

for name, df in [('Stage 1', s1), ('Stage 2', s2), ('Stage 3', s3)]:
    feat_cols = [c for c in df.columns if c not in
                 {'api_player_id','season','tm_player_id','market_value_in_eur','log_market_value'}]
    print(f"{name}: {len(df)} player-seasons, {len(feat_cols)} features")
    print(f"  Features: {feat_cols}\n")

---
## 5. Models

### Models used

| Model | Role | Notes |
|---|---|---|
| **Linear Regression** | Baseline | No regularisation — expected to overfit on Stage 2/3 |
| **Ridge Regression** | Regularised baseline | L2 penalty (α=10) stabilises correlated per-90 features |
| **Random Forest** | Main model | 200 trees, handles non-linearity and feature interactions well |
| **XGBoost** | Main model | Gradient boosting — typically best single model in tabular tasks |

### Evaluation protocol

**With a single season (current state):** we use **5-fold cross-validation** — the data is split into 5 folds, models trained on 4 and evaluated on the held-out fold, rotated 5 times. This gives an unbiased performance estimate but is slightly optimistic since all data is from the same season.

**With multiple seasons (once 2023/2024 data is fetched):** we switch to a **temporal train/test split** — train on 2022/23 + 2023/24, test on 2024/25. This is the correct evaluation for time-series data; random splits would leak future information.

### Metrics
- **R²**: proportion of variance explained — the headline number, target is > 0.80 (paper benchmark)
- **RMSE_log / MAE_log**: error in log-transformed space — model-comparable
- **MAE_eur**: mean absolute error back-transformed to euros — human-interpretable ("off by €X million on average")
- **Spearman ρ**: rank correlation — measures whether the model correctly orders players by value, even if absolute predictions are off

In [ ]:
from src.models import train_global, train_position_aware, results_to_df

all_results = []
for stage_name, df in [('stage1', s1), ('stage2', s2), ('stage3', s3)]:
    print(f'\n--- {stage_name} ---')
    all_results += train_global(df, stage_name)
    all_results += train_position_aware(df, stage_name)

table = results_to_df(all_results)
table

### 5a. R² by feature stage — global model

This is the main results chart. Each group of bars is one feature stage; the four bars within each group are the four models. The red dashed line marks the R²=0.80 benchmark from the reference paper.

**What to expect:** Tree models (RF, XGBoost) should improve or stay flat as we add more features. Linear models may degrade at Stage 2 due to correlated per-90 stats. The gap between tree models and linear models widens as feature complexity increases — evidence that the value function is non-linear.

In [ ]:
global_df = table[table['position'] == 'global'].copy()
stages = global_df['stage'].unique()
models = global_df['model'].unique()

fig, ax = plt.subplots(figsize=(10, 5))
x = np.arange(len(stages))
width = 0.2

for i, model in enumerate(models):
    sub = global_df[global_df['model'] == model]
    r2_vals = [sub[sub['stage'] == s]['R²'].values[0] if len(sub[sub['stage'] == s]) else 0 for s in stages]
    ax.bar(x + i * width, r2_vals, width, label=model)

ax.set_xticks(x + width * 1.5)
ax.set_xticklabels(stages)
ax.set_ylabel('R²  (higher is better)')
ax.set_title('R² by feature stage — global model\n(5-fold CV on 2022/23 season)')
ax.axhline(0.8, color='red', linestyle='--', linewidth=1.0, label='Paper benchmark (R²=0.80)')
ax.legend(fontsize=8)
ax.set_ylim(0, 1)
plt.tight_layout()
plt.show()

### 5b. Position-aware vs global — Random Forest, Stage 1

One of our research questions is whether training **separate models per position** (DEF / MID / FWD) outperforms a single global model. The reference paper finds it does.

The global model (blue) sees all positions together. The position-specific models (orange) each see only their position group. If the orange bars are higher, position-aware training is adding value — each model learns the specific value drivers for that position without being confused by the different scales of other positions.

In [ ]:
rf_s1 = table[(table['model'] == 'RandomForest') & (table['stage'] == 'stage1')].copy()

fig, ax = plt.subplots(figsize=(7, 4))
colors = ['steelblue' if p == 'global' else 'darkorange' for p in rf_s1['position']]
bars = ax.bar(rf_s1['position'], rf_s1['R²'], color=colors)
ax.bar_label(bars, fmt='%.3f', padding=3, fontsize=9)
ax.axhline(0.8, color='red', linestyle='--', linewidth=0.9, label='Paper benchmark (R²=0.80)')
ax.set_ylabel('R²')
ax.set_title('Random Forest Stage 1 — global vs position-aware R²')
ax.legend()
ax.set_ylim(0, 1)
plt.tight_layout()
plt.show()

### 5c. Mean Absolute Error in EUR

R² tells us how much variance the model explains, but MAE in euros is what matters practically: "on average, our prediction is off by €X million."

For context, a MAE of €7M on a dataset with a median value of €12M means we're within one order of magnitude for most players, but will be more wrong on extreme values (star players worth €100M+). This is expected — extreme values are the hardest to predict from statistics alone.

In [ ]:
g_s1 = table[(table['position'] == 'global') & (table['stage'] == 'stage1')].copy()

fig, ax = plt.subplots(figsize=(7, 4))
bars = ax.bar(g_s1['model'], g_s1['MAE_eur_M'], color='steelblue', edgecolor='white')
ax.bar_label(bars, fmt='€%.1fM', padding=3, fontsize=9)
ax.set_ylabel('MAE (€M)  (lower is better)')
ax.set_title('Mean Absolute Error in EUR — global model, Stage 1')
ax.set_ylim(0, max(g_s1['MAE_eur_M']) * 1.2)
plt.tight_layout()
plt.show()

### 5d. Full results table

Complete model × stage × position results. Use this as the basis for the main results table in the report.

**Key columns:**
- `n` — number of player-seasons evaluated (test set size with temporal split, full CV set with single season)
- `R²` — headline metric, log-space
- `Spearman_ρ` — rank correlation — useful when absolute predictions are noisy but ordering is correct
- `MAE_eur_M` — mean absolute error in millions of euros

In [ ]:
# Filter to tree models only (linear models noted separately in report)
tree_results = table[table['model'].isin(['RandomForest', 'XGBoost'])]
tree_results.sort_values(['stage', 'position', 'model']).reset_index(drop=True)

---
## 6. Player lookup — predicted vs actual value

Train the best model (Random Forest, Stage 1) on the full dataset and inspect predictions for individual players. Useful for sanity-checking and for the "where does the model deviate from Transfermarkt?" analysis in the report.

In [ ]:
from sklearn.ensemble import RandomForestRegressor
from src.models import _DROP_COLS
from src.features import _aggregate_multi_club, _add_age, _normalize_position

# Train RF on the full Stage 1 dataset
feature_cols = [c for c in s1.columns if c not in _DROP_COLS]
X = s1[feature_cols].fillna(0).values
y = s1['log_market_value'].values

rf = RandomForestRegressor(n_estimators=200, max_depth=8, min_samples_leaf=3,
                           random_state=42, n_jobs=-1)
rf.fit(X, y)

# Build predictions on Stage 1 rows
pred_df = s1.copy()
pred_df['predicted_log'] = rf.predict(pred_df[feature_cols].fillna(0).values)
pred_df['predicted_eur'] = pred_df['predicted_log'].apply(lambda x: round(np.expm1(x) / 1e6, 1))
pred_df['actual_eur']    = (pred_df['market_value_in_eur'] / 1e6).round(1)
pred_df['error_eur']     = (pred_df['predicted_eur'] - pred_df['actual_eur']).round(1)

# Attach names directly from joined (one row per player-season after dedup)
names = (
    joined[['api_player_id', 'season', 'firstname', 'lastname']]
    .drop_duplicates(['api_player_id', 'season'])
)
lookup = pred_df.merge(names, on=['api_player_id', 'season'], how='left')
lookup['name'] = lookup['firstname'].fillna('') + ' ' + lookup['lastname'].fillna('')

print(f"Predictions ready for {len(lookup)} players.")
print("Sample names:", lookup['name'].dropna().head(5).tolist())

In [ ]:
# --- Change the name below to look up any player ---
SEARCH = "Salah"

cols = ['name', 'season', 'age', 'minutes', 'goals', 'assists',
        'actual_eur', 'predicted_eur', 'error_eur']

result = lookup[lookup['name'].str.contains(SEARCH, case=False, na=False)][cols]
result = result.rename(columns={
    'actual_eur': 'actual (€M)',
    'predicted_eur': 'predicted (€M)',
    'error_eur': 'error (€M)',
})
result